In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第1周-Day1：自注意力机制详解

> **Welcome to Transformer 的世界！** 这是 13 周大模型学习之旅的第一步。今天我们从最核心的概念——Self-Attention（自注意力机制）开始。这个机制是所有现代大语言模型（GPT、BERT、LLaMA、GLM）的基石。理解了它，你就理解了 Transformer 的灵魂。


## 📅 学习进度


In [ ]:
W1 ████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░  ← 你在这里
W2 ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
W3 ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
...
W13 ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


**当前位置：W1 Day 1/7（13 周计划的第一周第一天）**

---


## 一、为什么需要自注意力机制？

### 🤔 一个问题：计算机怎么理解"意思"？

假设你看到这句话：

> "小明喜欢AI，因为**他**对编程感兴趣"

作为人类，你立刻知道"他"指的是"小明"。但计算机怎么知道？

传统方法（RNN/LSTM）是**从左到右逐词读**，像看书一行一行扫。读到"他"的时候，模型需要"记住"前面有个"小明"。但句子一长，前面的信息就忘了——就像你读到 chapter 10，已经忘了 chapter 1 的内容。

### 💡 自注意力的思路：让每个词同时"看到"所有其他词

2017 年，Google 的论文《Attention Is All You Need》提出了一个革命性想法：

> 不要逐个读了！让每个词同时看整句话，自己决定该关注谁。

**打个比方**：

想象你在一个派对上（社交场合）。你不是逐个和每个人聊天，而是同时扫视全场：
- 你看到穿红衣服的小明 → "嗯，他好像和我有共同话题"
- 你看到角落的小红 → "她看起来不太相关"
- 于是你把更多注意力放在小明身上

**自注意力做的就是这件事**：每个词都在"扫视"整句话，然后根据相关性分配注意力。

### 🔥 为什么这比传统方法好？

| 对比项 | RNN/LSTM | Self-Attention |
|--------|----------|----------------|
| 信息获取 | 逐词传递，远距离信息会衰减 | 任意两个词直接相连，距离不影响 |
| 并行计算 | 必须按顺序算，无法并行 | 所有词同时计算，完美并行 |
| 长文本 | 容易忘记前面的内容 | 每个词都能直接"看到"所有其他词 |

---


## 二、核心原理详解

### 2.1 Q/K/V：三个关键角色

自注意力最核心的设计是：每个词被映射成三个向量——**Q（Query）、K（Key）、V（Value）**。

#### 🎯 生活类比：搜索引擎

你每天用 Google 搜索，背后就是 Q/K/V 的逻辑：

| 角色 | 搜索比喻 | 在注意力中的作用 |
|------|---------|----------------|
| **Q（Query）查询** | 你在搜索框输入的关键词 | "我想找什么样的信息？" |
| **K（Key）键** | 网页的标题和标签 | "我能提供什么样的信息？" |
| **V（Value）值** | 网页的实际内容 | "我具体的信息内容是什么" |
| **注意力分数** | 搜索结果的相关度排序 | Q 和 K 越匹配，分数越高 |

**流程**：Q（你想找的）匹配 K（别人能提供的）→ 得到相关度分数 → 按分数提取 V（实际内容）。

#### 📐 数学表达

每个词的输入向量 X（比如词嵌入），分别乘以三个权重矩阵，得到 Q、K、V：


In [ ]:
Q = X × W_q    （查询向量："我想找什么"）
K = X × W_k    （键向量："我能提供什么"）
V = X × W_v    （值向量："我的实际内容"）


其中 W_q、W_k、W_v 是模型在训练过程中**学出来的**参数矩阵。

### 2.2 注意力分数：Q 和 K 怎么"匹配"？

匹配的方式出奇简单：**点积（Dot Product）**。


In [ ]:
注意力分数 = Q · K^T


**为什么点积能表示相关性？**

**打个比方**：两个向量就像两个人的兴趣标签。如果标签方向一致（点积大），说明兴趣相似、相关性高。如果方向相反（点积小或为负），说明不太相关。

具体来说，假设句子有 3 个词："我 爱 AI"

- "我"的 Q 去和"我"的 K 做点积 → 得到一个分数
- "我"的 Q 去和"爱"的 K 做点积 → 得到一个分数
- "我"的 Q 去和"AI"的 K 做点积 → 得到一个分数

这样得到一个 3×3 的分数矩阵。**分数越高，两个词越相关**。

### 2.3 缩放（Scaling）：为什么要除以 √d_k？

原始分数可能很大（尤其维度高的时候），直接做 softmax 会导致某些值无限大、其他值趋近于零——就像班级里一个学生考 100 分，其他人都考 10 分，差距太大不好分析。

**解决方法**：除以 √d_k（d_k 是 K 的维度）。


In [ ]:
缩放分数 = Q · K^T / √d_k


**打个比方**：考试满分从 100 分"缩放"到 10 分制，方便比较。这不改变排名（谁高谁低没变），只是让数值更合理。

### 2.4 Softmax：把分数变成概率

Softmax 的作用是把任意大小的数字变成 **0 到 1 之间的概率**，且所有概率之和等于 1。


In [ ]:
attention_weights = softmax(缩放分数)


**打个比方**：你有 100% 的注意力要分配。Softmax 决定了：
- 30% 注意力给"我"
- 50% 注意力给"爱"
- 20% 注意力给"AI"

这样，每个词都有一个"注意力预算分配方案"。

### 2.5 加权求和：用注意力分数提取信息

最后一步：用注意力分数对 V（值）做加权平均。


In [ ]:
输出 = 注意力权重 × V


**这就是自注意力的最终输出**——每个词的新表示，不再是自己一个人的信息，而是融合了整句话中相关信息的"富信息向量"。

### 2.6 完整公式

把上面所有步骤合在一起：


In [ ]:
Attention(Q, K, V) = softmax(Q · K^T / √d_k) · V


**用一句话概括**：每个词通过 Q 找到相关的 K，用分数对 V 做加权平均，得到融合了全文信息的新表示。

---


## 三、代码实战

让我们用纯 NumPy，一步一步实现 Self-Attention：


In [ ]:
import numpy as np

np.random.seed(42)

# ========== 第1步：准备输入 ==========
# 模拟句子 "我 爱 AI" —— 3个词，每个词4维向量
X = np.array([
    [1.0, 0.0, 0.0, 0.0],   # "我"的词向量
    [0.0, 1.0, 0.0, 0.0],   # "爱"的词向量
    [0.0, 0.0, 1.0, 0.5],   # "AI"的词向量
])

seq_len, d_model = X.shape
print(f"句子长度: {seq_len}, 向量维度: {d_model}")

# ========== 第2步：生成 Q, K, V ==========
# 权重矩阵是模型训练出来的，这里用随机值模拟
W_q = np.random.randn(d_model, d_model)  # 4×4 矩阵
W_k = np.random.randn(d_model, d_model)
W_v = np.random.randn(d_model, d_model)

# 矩阵乘法得到 Q, K, V
Q = X @ W_q   # 查询：每个词"想找什么"
K = X @ W_k   # 键：  每个词"能提供什么"
V = X @ W_v   # 值：  每个词"的实际内容"

print(f"Q shape: {Q.shape}")  # (3, 4) —— 3个词，每个4维

# ========== 第3步：计算注意力分数 ==========
# Q 和 K^T 做点积，得到 3×3 的分数矩阵
scores = Q @ K.T
print("原始注意力分数:")
print(np.round(scores, 3))
# 分数越高 → 两个词越相关

# ========== 第4步：缩放 + Softmax ==========
d_k = d_model
scaled_scores = scores / np.sqrt(d_k)  # 除以 √4 = 2

def softmax(x):
    """数值稳定的 softmax：先减最大值防止溢出"""
    x_shifted = x - x.max(axis=1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

attn_weights = softmax(scaled_scores)
print("\n注意力权重（softmax后，每行和=1）:")
print(np.round(attn_weights, 3))

# ========== 第5步：加权求和 ==========
output = attn_weights @ V
print("\n最终输出:")
print(np.round(output, 3))
# 每个词的输出 = 所有词 V 的加权平均
# 权重 = 注意力分数


**运行结果解读**：

注意力权重矩阵中，`attn_weights[0][1]` 表示"我"对"爱"的关注度。权重越高，说明模型认为这两个词越相关。最终输出中，每个词的向量都融合了整句话的信息。

---


## 四、可视化理解

用 matplotlib 画出注意力权重的热力图，直观理解"谁在关注谁"：


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

# 中文字体配置（必须！否则中文会显示为方框）
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 假设这是计算好的注意力权重
attn_weights = np.array([
    [0.20, 0.50, 0.30],  # "我" 对 [我, 爱, AI] 的注意力
    [0.15, 0.45, 0.40],  # "爱" 对 [我, 爱, AI] 的注意力
    [0.25, 0.35, 0.40],  # "AI" 对 [我, 爱, AI] 的注意力
])

words = ["我", "爱", "AI"]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_weights, cmap='Blues')

# 设置坐标轴
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, fontsize=14)
ax.set_yticklabels(words, fontsize=14)
ax.set_xlabel('被关注的词 (Key)', fontsize=12)
ax.set_ylabel('发起关注的词 (Query)', fontsize=12)
ax.set_title('自注意力权重热力图', fontsize=14)

# 在每个格子里写上数值
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f'{attn_weights[i][j]:.2f}',
                ha='center', va='center', fontsize=14,
                color='white' if attn_weights[i][j] > 0.4 else 'black')

plt.colorbar(im, label='注意力权重')
plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=150)
plt.show()
print("图表已保存为 attention_heatmap.png")


**如何看热力图**：颜色越深（越蓝），说明注意力权重越高。比如"他"这一行，如果"小明"那列颜色最深，说明模型认为"他"指代的就是"小明"。

---


## 五、业务关联

### 🏪 自注意力和 LangChat/Agent 有什么关系？

你可能会问：学这个对做 AI 应用有什么用？

**直接关系**：

1. **所有大模型都基于 Self-Attention**：你用的 ChatGPT、GLM、文心一言，底层都是 Transformer，而 Transformer 的核心就是 Self-Attention。理解它，你才知道为什么模型有时候"聪明"，有时候"犯傻"。

2. **Agent 的上下文窗口**：当你给 Agent 发一条很长的指令时，Self-Attention 决定了模型如何理解你的意图。如果注意力分配不好，模型可能"看不到"关键信息。

3. **企业 AI 搜索**：做企业知识库搜索时，理解注意力机制能帮你设计更好的 Prompt——把重要信息放在显眼的位置，帮助模型分配更多注意力。

4. **糖水店经营分析**：假设你做一个 AI 经营助手，让它分析"这个月为什么销量下降"。Self-Attention 让模型能同时关联天气、促销、竞品、季节等多维度信息，而不是逐个看。

**Jason 的实际场景**：在构建 LangChat 时，用户可能发一段 500 字的需求描述。模型需要理解这段话的核心意图——注意力机制就是让模型"抓住重点"的关键技术。

---


## 六、常见误区

### ❌ 误区1："Q、K、V 是三个不同的模型"

**纠正**：Q、K、V 是同一个输入 X 经过不同权重矩阵变换后的三个"面"。就像同一个人，在面试时展示"技能"（Q）、在简历上写"标签"（K）、实际工作中产出"成果"（V）。

### ❌ 误区2："注意力权重越高越好"

**纠正**：不是！如果一个词对所有词的注意力都是均匀的（各 1/n），说明它没找到相关信息。但如果某个词 100% 只关注自己，可能又太"自恋"了。好的注意力分布应该是**有选择性**的——该高的时候高，该低的时候低。

### ❌ 误区3："Self-Attention 就是 RNN 的升级版"

**纠正**：它们是完全不同的架构。RNN 是**串行**的（一个接一个处理），Self-Attention 是**并行**的（所有词同时处理）。这不仅是效率差异，更是建模思路的根本改变——从"逐字阅读"到"全局扫视"。

---


## 🧪 课堂练习（5分钟）

**题目1**：给定以下 Q 和 K 向量，手动计算注意力分数（不需要 softmax）：


In [ ]:
Q = [1, 0]
K1 = [1, 1]
K2 = [0, 1]


提示：点积 = 对应位置相乘再求和。

**题目2**：如果一句话有 100 个词，注意力分数矩阵是多大？需要计算多少次点积？

**题目3**：为什么说 Self-Attention 解决了"长距离依赖"问题？用一句话解释。

---


## 📝 课后测试（15分钟）

**第1题（选择）**：在 Self-Attention 中，Q 和 K 的主要作用是？
- A) Q 存储信息，K 提取特征
- B) Q 用于查询匹配，K 用于被查询匹配
- C) Q 和 K 是完全相同的东西
- D) Q 决定输出，K 决定输入

**第2题（填空）**：Self-Attention 的完整公式是 Attention(Q,K,V) = ______。

**第3题（简答）**：为什么注意力分数要除以 √d_k？不除会怎样？

**第4题（简答）**：假设你在构建一个客服 Agent，用户说"帮我查一下上周的订单，对了我的会员等级是什么"。Self-Attention 如何帮助模型理解这句话？

**第5题（思考）**：如果 Q = K（即权重矩阵相同），注意力分数矩阵会变成什么？这对模型的学习有什么影响？

---


## 🔑 今日术语

| 英文 | 音标 | 中文解释 |
|------|------|---------|
| Self-Attention | [sɛlf əˈtɛnʃən] | 自注意力机制，让每个词"看到"所有其他词 |
| Query (Q) | [ˈkwɪəri] | 查询向量，表示"我想找什么信息" |
| Key (K) | [kiː] | 键向量，表示"我能提供什么信息" |
| Value (V) | [ˈvæljuː] | 值向量，表示"我的具体内容" |
| Dot Product | [dɒt ˈprɒdʌkt] | 点积，两个向量对应位置相乘再求和 |
| Softmax | [ˈsɒftmæks] | 归一化函数，把任意数值变成概率分布 |
| Scaling | [ˈskeɪlɪŋ] | 缩放，除以 √d_k 防止分数过大 |
| Weight Matrix | [weɪt ˈmeɪtrɪks] | 权重矩阵，模型训练学习的参数 |

---


## 📎 参考资源

### 📄 必读论文
- **Attention Is All You Need** (Vaswani et al., 2017)
  - 论文地址：https://arxiv.org/abs/1706.03762
  - 这是 Transformer 的开山之作，必读！

### 🎬 推荐视频
- ⭐ **3Blue1Brown - 直观解释注意力机制**（全球公认最优秀的可视化讲解）
  - https://www.bilibili.com/video/BV1TZ421j7Ke/
- 📺 **15分钟认识注意力机制**（B站·数学原理详解）
  - https://www.bilibili.com/video/BV1pj42137ZY/
- 🎓 **李沐 - Attention Is All You Need 论文精读**
  - https://www.bilibili.com/video/BV1pu411o7BE/

### 📖 延伸阅读
- ⭐ **Jay Alammar - The Illustrated Transformer**（全网最经典图解，必看！）
  - https://jalammar.github.io/illustrated-transformer/
- 📝 **图解 Transformer：深入理解 Self-Attention**（知乎）
  - https://zhuanlan.zhihu.com/p/651018724

### 💻 代码参考
- **The Annotated Transformer**（Harvard NLP，带详细注释的实现）
  - http://nlp.seas.harvard.edu/2018/04/03/attention.html

### 📁 相关 Notebook
- `第1周/第1周-Day1-自注意力.ipynb` —— 对应的代码练习

---

> 💡 **进度：W1 Day 1/7 | 🤖 大模型基础 | 下一篇：Day 2 - 多头注意力与位置编码**
